# SAFOD solid-Earth tides: forcing, Thomas-style stress, and response models

This notebook is the **presentation layer** for the Sherlock pipeline. The external tide packages and response models are executed by the scripts in `scripts/tides/`; this notebook reads the products in `outputs/tides/`.

To regenerate everything on Sherlock:

```bash
git pull
bash scripts/tides/setup_sherlock.sh      # first-time environment/SPOTL setup
bash RUN_ON_SHERLOCK.sh
```

Then select the Jupyter kernel **SAFOD tides (.venv)** and run this notebook.

The modeling chain is

$$
\text{Sun/Moon}
\rightarrow
\boldsymbol{\varepsilon}(t)
\rightarrow
\boldsymbol{\sigma}(t)
\rightarrow
\{\mathrm{FNS}(t),\mathrm{RLSS}(t)\}
\rightarrow
\Delta v/v.
$$

The first arrow is constrained with PySolid, SPOTL, and a transparent degree-2 calculation. The later arrows contain the more consequential constitutive assumptions.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from IPython.display import display

def find_root():
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "config.json").exists() and (p / "scripts/tides").exists():
            return p
    raise FileNotFoundError("Could not locate project root.")

ROOT = find_root()
OUT = ROOT / "outputs/tides"
CONFIG = json.loads((ROOT / "config.json").read_text())

print("Project root:", ROOT)
print("Results:", OUT)


## 1. Did the external calculations actually run?

A package is treated as a package result only if its numerical output **and** provenance file exist. The notebook never substitutes an analytic surrogate for a missing package calculation.


In [ ]:
def load_json(path):
    p = Path(path)
    return json.loads(p.read_text()) if p.exists() else None

rows=[]
for name,csv_name,prov_name in [
    ("PySolid","pysolid_tides.csv","pysolid_provenance.json"),
    ("SPOTL ertid","spotl_ertid_tides.csv","spotl_provenance.json"),
    ("analytic degree-2","analytic_degree2_tides.csv","analytic_degree2_provenance.json"),
    ("Models A-D","model_results.csv","model_provenance.json"),
]:
    p=OUT/csv_name
    q=OUT/prov_name
    prov=load_json(q)
    rows.append({
        "product":name,
        "data":p.exists(),
        "provenance":q.exists(),
        "hostname":None if prov is None else prov.get("hostname"),
        "created_utc":None if prov is None else prov.get("created_utc"),
        "git_commit":None if prov is None else prov.get("git_commit"),
    })
display(pd.DataFrame(rows))


## 2. Tide forcing

**PySolid** returns solid-Earth-tide displacement. The Sherlock script evaluates a small spatial stencil around SAFOD and differentiates the displacement field to obtain the horizontal strain tensor.

**SPOTL `ertid`** can return extensional strain directly. The script requests strain at $0^\circ$, $45^\circ$, and $90^\circ$ and reconstructs

$$
\varepsilon_{NN},\qquad
\varepsilon_{EE},\qquad
\varepsilon_{NE}.
$$

The SPOTL manual states that `ertid` computes body tides directly from Sun/Moon positions and reports strain in nanostrain, positive for extension.

References: Agnew (2012), SPOTL; Métivier & Conrad (2008), DOI `10.1029/2007JB005448`.


In [ ]:
def read_product(name):
    p=OUT/name
    if not p.exists():
        return None
    return pd.read_csv(p,parse_dates=["time_utc"])

pysolid=read_product("pysolid_tides.csv")
spotl=read_product("spotl_ertid_tides.csv")
analytic=read_product("analytic_degree2_tides.csv")

print("PySolid:", "not present" if pysolid is None else len(pysolid))
print("SPOTL:", "not present" if spotl is None else len(spotl))
print("analytic:", "not present" if analytic is None else len(analytic))


In [ ]:
if pysolid is None:
    print("No PySolid result yet. Run bash RUN_ON_SHERLOCK.sh")
else:
    plt.figure(figsize=(11,5))
    plt.plot(pysolid.time_utc,1e9*(pysolid.areal_strain-pysolid.areal_strain.mean()),
             linewidth=2,label="PySolid")
    if spotl is not None:
        plt.plot(spotl.time_utc,1e9*(spotl.areal_strain-spotl.areal_strain.mean()),
                 label="SPOTL ertid")
    if analytic is not None and "areal_strain" in analytic.columns:
        plt.plot(analytic.time_utc,1e9*(analytic.areal_strain-analytic.areal_strain.mean()),
                 label="transparent degree-2")
    plt.axhline(0,linewidth=.8)
    plt.ylabel("Mean-removed areal strain (nanostrain)")
    plt.xlabel("UTC")
    plt.title("SAFOD body-tide forcing, June 16–17 2026")
    plt.grid(alpha=.25); plt.legend()
    plt.gca().xaxis.set_major_locator(mdates.HourLocator(interval=3))
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter("%m-%d %H:%M"))
    plt.xticks(rotation=30,ha="right"); plt.tight_layout(); plt.show()


In [ ]:
comparison_path=OUT/"forcing_comparison.csv"
if comparison_path.exists():
    display(pd.read_csv(comparison_path))
else:
    print("Package-comparison table will appear after both PySolid and SPOTL have run.")


# 3. Corrected Model B: Thomas-style tidal stress benchmark

Thomas et al. (2012, DOI `10.1029/2011JB009036`) used the following architecture near Parkfield:

1. calculate tidal strain with SPOTL;
2. convert strain to stress with a **linear elastic constitutive equation**;
3. resolve the stress onto a **vertical** San Andreas fault plane striking N42°W;
4. examine fault-normal stress (FNS) and right-lateral shear stress (RLSS).

They state that the solid-Earth tide is sufficiently long wavelength that the **surface body-tide strain is not significantly different from the strain at 25 km depth**. They do **not** state that they assume “plane stress.”

### What was wrong with the previous Model B?

The previous implementation directly called the constitutive step *plane stress* and then projected that surface stress tensor onto a dipping SAFOD plane. That conflated two separate issues:

- the free-surface boundary condition used to complete an incomplete surface strain tensor;
- the unknown three-dimensional stress tensor at the $\sim1$ km DAS depth.

The corrected Model B no longer claims that the subsurface SAFOD problem is plane stress and no longer projects the surface stress tensor onto the $\sim70^\circ$ dipping plane.

### Explicit surface closure

For the package outputs available here we know

$$
\varepsilon_{NN},\quad
\varepsilon_{EE},\quad
\varepsilon_{NE}.
$$

At the free surface, assuming isotropic linear elasticity,

$$
\sigma_{UU}=0.
$$

With

$$
\sigma_{ij}
=
\lambda\,\varepsilon_{kk}\delta_{ij}
+
2\mu\varepsilon_{ij},
$$

this gives

$$
\varepsilon_{UU}
=
-\frac{\lambda}{\lambda+2\mu}
(\varepsilon_{NN}+\varepsilon_{EE})
=
-\frac{\nu}{1-\nu}
(\varepsilon_{NN}+\varepsilon_{EE}).
$$

We then construct the **full surface stress tensor with 3-D Hooke's law**. Algebraically, its horizontal components are identical to the familiar plane-stress formulas, but the logic is now explicit: this is a **surface boundary-condition closure**, not a claim that the rock at 1 km is a plane-stress body.

Because the Thomas benchmark uses a **vertical fault**, FNS and strike-parallel shear depend only on the horizontal stress components. That makes this a clean published-style benchmark without pretending that we know the full stress tensor on the dipping SAFOD fault at depth.

A useful independent precedent is Bucholc & Steacy (2016, DOI `10.1093/gji/ggw045`), who explicitly use the full 3-D isotropic relation

$$
\boldsymbol{\sigma}
=
\lambda(\nabla\cdot\mathbf u)\mathbf I
+
2\mu\boldsymbol{\varepsilon}
$$

and obtain radial strain from volumetric strain before resolving tidal stresses onto faults.


## 3.1 Reproduce the logic of Thomas et al. Figure 3 for the SAFOD experiment window

Thomas et al. Figure 3 shows a representative 14-day time series of:

- **FNS**: fault-normal stress, positive for tension/unclamping;
- **RLSS**: right-lateral shear stress.

Here we make the analogous plot for **June 16–17, 2026 at SAFOD**. When SPOTL is available, the plot uses SPOTL because that most closely follows Thomas et al.; otherwise it shows the PySolid forcing.

The fault plane is vertical and strikes N42°W ($318^\circ$), exactly matching the geometry stated by Thomas et al. This is extremely close to the local SAF strike direction but is used here specifically as a methodological reproduction.


In [ ]:
model_path=OUT/"model_results.csv"
models=pd.read_csv(model_path,parse_dates=["time_utc"]) if model_path.exists() else None

if models is None:
    print("No model results yet. Run bash RUN_ON_SHERLOCK.sh")
else:
    forcing = "spotl" if f"spotl_FNS_tension_pa" in models.columns else "pysolid"
    fns_col=f"{forcing}_FNS_tension_pa"
    rlss_col=f"{forcing}_RLSS_right_lateral_pa"

    if fns_col not in models.columns:
        print("Your model_results.csv predates the corrected Model B. Run git pull and rerun the pipeline.")
    else:
        good=models[[ "time_utc",fns_col,rlss_col ]].dropna()

        plt.figure(figsize=(11,5))
        # Original Thomas Fig. 3 uses blue FNS and red RLSS.
        plt.plot(good.time_utc,good[fns_col],color="tab:blue",label="FNS: normal stress (+ tension)")
        plt.plot(good.time_utc,good[rlss_col],color="tab:red",label="RLSS: right-lateral shear")
        plt.axhline(0,color="0.4",linewidth=.8)
        plt.ylabel("Tidal stress (Pa)")
        plt.xlabel("UTC")
        plt.title(f"Thomas et al. (2012) Fig. 3 analogue — SAFOD, {forcing}")
        plt.legend(); plt.grid(alpha=.25)
        plt.gca().xaxis.set_major_locator(mdates.HourLocator(interval=3))
        plt.gca().xaxis.set_major_formatter(mdates.DateFormatter("%m-%d %H:%M"))
        plt.xticks(rotation=30,ha="right"); plt.tight_layout(); plt.show()

        Af=np.max(np.abs(good[fns_col]))
        As=np.max(np.abs(good[rlss_col]))
        print(f"forcing: {forcing}")
        print(f"max |FNS|  = {Af:.2f} Pa")
        print(f"max |RLSS| = {As:.2f} Pa")
        print(f"|FNS|/|RLSS| amplitude ratio = {Af/As:.2f}")


Thomas et al. report that the solid-Earth tides produce largely volumetric stresses and that RLSS is approximately an order of magnitude smaller than FNS. The ratio printed above is therefore a useful **qualitative diagnostic** of whether our stress construction behaves like their Parkfield calculation.

It is not an exact numerical reproduction of Thomas et al. because our site, dates, tide implementation, and elastic constants are different.


## 3.2 What Model B does — and does not — predict

The corrected Model B tidal-stress output is

$$
\mathrm{FNS}(t),\qquad \mathrm{RLSS}(t).
$$

Only after that do we make a separate empirical transfer assumption:

$$
\left(\frac{\Delta v}{v}\right)_B
=
S_{\mathrm{Niu}}\,\mathrm{FNS}(t),
$$

with

$$
S_{\mathrm{Niu}}=2.4\times10^{-7}\ {\rm Pa}^{-1}.
$$

This coefficient came from Niu et al. (2008)'s SAFOD barometric loading experiment. **Niu did not establish that the same coefficient applies to tidal FNS.** That transfer is ours and remains a model assumption.

The corrected Model B therefore separates two questions:

$$
\boxed{\text{tide strain}\rightarrow\text{tidal FNS/RLSS}}
$$

from

$$
\boxed{\text{tidal stress}\rightarrow\Delta v/v}.
$$

That separation is important because the first can be checked against Thomas-style tidal-stress calculations even if the second remains uncertain.


# 4. Models A–D


In [ ]:
if models is None:
    print("No model products yet.")
else:
    rows=[]
    for forcing in ["pysolid","spotl"]:
        mapping={
            "A":f"{forcing}_model_A_dv_over_v",
            "B (Niu × FNS)":f"{forcing}_model_B_dv_over_v",
            "C (Takano transfer)":f"{forcing}_model_C_takano_dv_over_v",
            "D (Vs crack model)":f"{forcing}_model_D_dVs_over_Vs",
            "D (Vp crack model)":f"{forcing}_model_D_dVp_over_Vp",
        }
        for name,col in mapping.items():
            if col in models.columns:
                rows.append({
                    "forcing":forcing,
                    "model":name,
                    "max_abs_dv_over_v":np.nanmax(np.abs(models[col])),
                    "max_abs_percent":100*np.nanmax(np.abs(models[col])),
                })
    summary=pd.DataFrame(rows)
    display(summary.style.format({
        "max_abs_dv_over_v":"{:.3e}",
        "max_abs_percent":"{:.5f}",
    }))


### Interpretation of the branches

**Model A** adopts Niu et al.'s published 240 Pa tidal-stress scale and applies Niu's local empirical stress sensitivity.

**Model B** now computes a Thomas-style FNS/RLSS stress time series first, then applies Niu's coefficient to FNS as an explicitly separate transfer assumption.

**Model C** applies published foreign-site strain sensitivities directly. It is context, not expected SAFOD behavior.

**Model D** makes stress alter crack density and elastic moduli before calculating $V_P$ and $V_S$. Its crack-closure scale is calibrated to the Niu local slope, so agreement with Model B is not independent validation.


## 5. AWD detectability

The strongest Deep outbound empirical reliable-tested change is

$$
\left|\frac{\Delta v}{v}\right|=5\times10^{-3}=0.5\%.
$$

This is a measurement benchmark, not a theoretical tide-detection threshold.


In [ ]:
if models is not None and 'summary' in globals() and len(summary):
    deep=CONFIG["awd_benchmarks"]["deep_outbound_reliable"]
    detect=summary.copy()
    detect["deep_reliable_over_model"]=deep/detect["max_abs_dv_over_v"]
    display(detect.style.format({
        "max_abs_dv_over_v":"{:.3e}",
        "max_abs_percent":"{:.5f}",
        "deep_reliable_over_model":"{:.1f}x",
    }))


# 6. What remains unresolved

The main remaining Model B limitation is no longer hidden:

> We do not yet have a rigorously computed full three-dimensional tidal strain/stress tensor at the $\sim1$ km SAFOD DAS depth.

Thomas et al. could use a vertical fault benchmark and argued that body-tide surface strain is nearly depth invariant over 25 km because of the enormous wavelength. For our dipping SAFOD geometry, however, a true traction calculation would require the radial and vertical-shear stress components as well.

A stronger future depth-aware Model B would obtain

$$
\varepsilon_{NN},\;
\varepsilon_{EE},\;
\varepsilon_{UU},\;
\varepsilon_{NE},\;
\varepsilon_{NU},\;
\varepsilon_{EU}
$$

at depth, apply

$$
\sigma_{ij}
=
\lambda\varepsilon_{kk}\delta_{ij}
+
2\mu\varepsilon_{ij},
$$

and only then resolve traction onto the dipping fault.

Until that exists, the **Thomas-style vertical-SAF FNS/RLSS calculation is the primary stress benchmark** and the dipping-plane stress projection is intentionally omitted.


# 7. References

- Thomas, A. M., Bürgmann, R., Shelly, D. R., Beeler, N. M., & Rudolph, M. L. (2012), *JGR Solid Earth*, DOI `10.1029/2011JB009036`.
- Agnew, D. C. (2012), *SPOTL: Some Programs for Ocean-Tide Loading*.
- Bucholc, M., & Steacy, S. (2016), *Geophysical Journal International*, DOI `10.1093/gji/ggw045`.
- Niu, F., Silver, P. G., Daley, T. M., Cheng, X., & Majer, E. L. (2008), *Nature*, DOI `10.1038/nature07111`.
- Boness, N. L., & Zoback, M. D. (2004), *GRL*, DOI `10.1029/2003GL019020`.
- Takano, T., et al. (2014), *GRL*, DOI `10.1002/2014GL060690`.
